# LWBS significance tests


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import mannwhitneyu, chi2_contingency
from pathlib import Path

### 1. LOAD AND VALIDATE DATA

In [2]:
PROJECT_ROOT = Path.cwd()
FILE_PATH = PROJECT_ROOT / "docs" / "ed_visits_timestamp_corrected.csv"

df = pd.read_csv(FILE_PATH)

df["Arrival_Timestamp"] = pd.to_datetime(
    df["Arrival_Timestamp"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

df["Observed_Triage_Wait_Minutes"] = pd.to_numeric(
    df["Observed_Triage_Wait_Minutes"],
    errors="coerce"
)

df["Triage_Score"] = pd.to_numeric(
    df["Triage_Score"],
    errors="coerce"
)

df["LWBS"] = (
    df["Disposition_Type"].str.strip().str.upper() == "LWBS"
).astype(int)

df = df.dropna(
    subset=[
        "Patient_ID",
        "Arrival_Timestamp",
        "Observed_Triage_Wait_Minutes",
        "Triage_Score"
    ]
).copy()

print("Rows analysed:", len(df))
print("Unique visits:", df["Visit_ID"].nunique())
print("Unique patients:", df["Patient_ID"].nunique())
print("Missing timestamps:", df["Arrival_Timestamp"].isna().sum())
print("LWBS cases:", df["LWBS"].sum())
print(f"LWBS rate: {df['LWBS'].mean() * 100:.2f}%")

Rows analysed: 200000
Unique visits: 200000
Unique patients: 49059
Missing timestamps: 0
LWBS cases: 19872
LWBS rate: 9.94%


### 2. LWBS BY TRIAGE SCORE

In [3]:
triage_summary = (
    df.groupby("Triage_Score")
      .agg(
          Visits=("Visit_ID", "count"),
          LWBS_Cases=("LWBS", "sum"),
          LWBS_Rate=("LWBS", "mean")
      )
      .reset_index()
)

triage_summary["LWBS_Rate_Percent"] = (
    triage_summary["LWBS_Rate"] * 100
).round(2)

display(triage_summary)

,Triage_Score,Visits,LWBS_Cases,LWBS_Rate,LWBS_Rate_Percent
0,1,20046,2017,0.100619,10.06
1,2,39846,3993,0.100211,10.02
2,3,79804,7924,0.099293,9.93
3,4,40204,3944,0.098100,9.81
4,5,20100,1994,0.099204,9.92


### 3. GEE: TRIAGE SCORE VS LWBS

In [4]:
triage_model = smf.gee(
    formula="LWBS ~ C(Triage_Score)",
    groups="Patient_ID",
    data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

triage_test = triage_model.wald_test_terms(scalar=True).table
triage_row = triage_test.loc["C(Triage_Score)"]

print("\nTRIAGE SCORE SIGNIFICANCE TEST")
print("Wald Chi-square:", round(float(triage_row["statistic"]), 4))
print("Degrees of freedom:", int(triage_row["df_constraint"]))
print("P-value:", round(float(triage_row["pvalue"]), 6))


TRIAGE SCORE SIGNIFICANCE TEST
Wald Chi-square: 1.39
Degrees of freedom: 4
P-value: 0.845935


### 4. LWBS BY WAIT-TIME BAND

In [5]:
bins = [-1, 15, 30, 45, 60]
labels = ["0-15", "16-30", "31-45", "46-60"]

df["Wait_Band"] = pd.cut(
    df["Observed_Triage_Wait_Minutes"],
    bins=bins,
    labels=labels
)

wait_summary = (
    df.groupby("Wait_Band", observed=False)
      .agg(
          Visits=("Visit_ID", "count"),
          LWBS_Cases=("LWBS", "sum"),
          LWBS_Rate=("LWBS", "mean")
      )
      .reset_index()
)

wait_summary["LWBS_Rate_Percent"] = (
    wait_summary["LWBS_Rate"] * 100
).round(2)

display(wait_summary)


,Wait_Band,Visits,LWBS_Cases,LWBS_Rate,LWBS_Rate_Percent
0,0-15,49350,4830,0.097872,9.79
1,16-30,51261,5138,0.100232,10.02
2,31-45,51365,5103,0.099348,9.93
3,46-60,48024,4801,0.099971,10.00


### 5. GEE: WAIT-TIME BAND VS LWBS

In [6]:
wait_model = smf.gee(
    formula="LWBS ~ C(Wait_Band)",
    groups="Patient_ID",
    data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

wait_test = wait_model.wald_test_terms(scalar=True).table
wait_row = wait_test.loc["C(Wait_Band)"]

print("\nWAIT-BAND GEE SIGNIFICANCE TEST")
print("Wald Chi-square:", round(float(wait_row["statistic"]), 4))
print("Degrees of freedom:", int(wait_row["df_constraint"]))
print("P-value:", round(float(wait_row["pvalue"]), 6))



WAIT-BAND GEE SIGNIFICANCE TEST
Wald Chi-square: 1.8564
Degrees of freedom: 3
P-value: 0.602733


### 6. PEARSON CHI-SQUARE: WAIT BAND VS LWBS

In [7]:
contingency = pd.crosstab(
    df["Wait_Band"],
    df["LWBS"]
)

chi2, chi_p, dof, expected = chi2_contingency(contingency)

print("\nWAIT-BAND PEARSON CHI-SQUARE TEST")
print("Chi-square:", round(chi2, 4))
print("Degrees of freedom:", dof)
print("P-value:", round(chi_p, 6))


WAIT-BAND PEARSON CHI-SQUARE TEST
Chi-square: 1.8565
Degrees of freedom: 3
P-value: 0.602712


### 7. MANN-WHITNEY U: WAIT DISTRIBUTIONS
#### LWBS vs non-LWBS

In [8]:
lwbs_wait = df.loc[
    df["LWBS"] == 1,
    "Observed_Triage_Wait_Minutes"
]

non_lwbs_wait = df.loc[
    df["LWBS"] == 0,
    "Observed_Triage_Wait_Minutes"
]

u_stat, mw_p = mannwhitneyu(
    lwbs_wait,
    non_lwbs_wait,
    alternative="two-sided"
)

print("\nMANN-WHITNEY U TEST")
print("LWBS median wait:", lwbs_wait.median())
print("Non-LWBS median wait:", non_lwbs_wait.median())
print("U statistic:", round(u_stat, 2))
print("P-value:", round(mw_p, 6))




MANN-WHITNEY U TEST
LWBS median wait: 30.0
Non-LWBS median wait: 30.0
U statistic: 1797490838.0
P-value: 0.316247


### 8. GEE: ARRIVAL YEAR VS LWBS

In [9]:
df["Arrival_Year"] = df["Arrival_Timestamp"].dt.year

year_summary = (
    df.groupby("Arrival_Year")
      .agg(
          Visits=("LWBS", "size"),
          LWBS_Cases=("LWBS", "sum"),
          LWBS_Rate=("LWBS", "mean")
      )
      .reset_index()
)

year_summary["LWBS_Rate_Percent"] = (
    year_summary["LWBS_Rate"] * 100
).round(2)

display(year_summary)

year_model = smf.gee(
    formula="LWBS ~ C(Arrival_Year)",
    groups="Patient_ID",
    data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

year_test = year_model.wald_test_terms(scalar=True).table
year_row = year_test.loc["C(Arrival_Year)"]

print("\nARRIVAL YEAR SIGNIFICANCE TEST")
print("Wald Chi-square:", round(float(year_row["statistic"]), 4))
print("Degrees of freedom:", int(year_row["df_constraint"]))
print("P-value:", round(float(year_row["pvalue"]), 6))


,Arrival_Year,Visits,LWBS_Cases,LWBS_Rate,LWBS_Rate_Percent
0,2019,28535,2776,0.097284,9.73
1,2020,28296,2806,0.099166,9.92
2,2021,28577,2883,0.100885,10.09
3,2022,28493,2795,0.098094,9.81
4,2023,28683,2800,0.097619,9.76
5,2024,28697,2932,0.102171,10.22
6,2025,28719,2880,0.100282,10.03



ARRIVAL YEAR SIGNIFICANCE TEST
Wald Chi-square: 6.4206
Degrees of freedom: 6
P-value: 0.377757


### 9. GEE: ARRIVAL HOUR VS LWBS

In [10]:
df["Arrival_Hour"] = df["Arrival_Timestamp"].dt.hour

hour_summary = (
    df.groupby("Arrival_Hour")
      .agg(
          Visits=("LWBS", "size"),
          LWBS_Cases=("LWBS", "sum"),
          LWBS_Rate=("LWBS", "mean")
      )
      .reset_index()
)

hour_summary["LWBS_Rate_Percent"] = (
    hour_summary["LWBS_Rate"] * 100
).round(2)

display(hour_summary)

hour_model = smf.gee(
    formula="LWBS ~ C(Arrival_Hour)",
    groups="Patient_ID",
    data=df,
    family=sm.families.Binomial(),
    cov_struct=sm.cov_struct.Exchangeable()
).fit()

hour_test = hour_model.wald_test_terms(scalar=True).table
hour_row = hour_test.loc["C(Arrival_Hour)"]

print("\nARRIVAL HOUR SIGNIFICANCE TEST")
print("Wald Chi-square:", round(float(hour_row["statistic"]), 4))
print("Degrees of freedom:", int(hour_row["df_constraint"]))
print("P-value:", round(float(hour_row["pvalue"]), 6))

,Arrival_Hour,Visits,LWBS_Cases,LWBS_Rate,LWBS_Rate_Percent
0,0,8320,790,0.094952,9.50
1,1,8513,826,0.097028,9.70
2,2,8270,816,0.098670,9.87
3,3,8307,849,0.102203,10.22
4,4,8395,813,0.096843,9.68
5,5,8232,839,0.101919,10.19
6,6,8457,874,0.103346,10.33
7,7,8231,776,0.094278,9.43
8,8,8397,787,0.093724,9.37
9,9,8328,868,0.104227,10.42



ARRIVAL HOUR SIGNIFICANCE TEST
Wald Chi-square: 30.8543
Degrees of freedom: 23
P-value: 0.126415


### 10. FINAL RESULTS SUMMARY

In [11]:
# --------------------------------------------------
# SECTION 10: CONSOLIDATED EXPLORATORY RESULTS
# --------------------------------------------------

import numpy as np
import pandas as pd
from pathlib import Path

# Build the results table
results_summary = pd.DataFrame([
    {
        "Analysis": "Triage Score vs LWBS (GEE)",
        "Chi_Square": float(triage_row["statistic"]),
        "DF": int(triage_row["df_constraint"]),
        "P_Value": float(triage_row["pvalue"])
    },
    {
        "Analysis": "Wait Band vs LWBS (GEE)",
        "Chi_Square": float(wait_row["statistic"]),
        "DF": int(wait_row["df_constraint"]),
        "P_Value": float(wait_row["pvalue"])
    },
    {
        "Analysis": "Wait Band vs LWBS (Pearson)",
        "Chi_Square": float(chi2),
        "DF": int(dof),
        "P_Value": float(chi_p)
    },
    {
        "Analysis": "Arrival Year vs LWBS (GEE)",
        "Chi_Square": float(year_row["statistic"]),
        "DF": int(year_row["df_constraint"]),
        "P_Value": float(year_row["pvalue"])
    },
    {
        "Analysis": "Arrival Hour vs LWBS (GEE)",
        "Chi_Square": float(hour_row["statistic"]),
        "DF": int(hour_row["df_constraint"]),
        "P_Value": float(hour_row["pvalue"])
    },
    {
        "Analysis": "Wait Distribution: LWBS vs Non-LWBS",
        "Chi_Square": np.nan,
        "DF": np.nan,
        "P_Value": float(mw_p)
    }
])

# Determine statistical significance at 5% level
results_summary["Significant_at_0.05"] = (
    results_summary["P_Value"] < 0.05
)

# Display results
display(results_summary.round(6))

# Save results using a portable relative path
OUTPUT_PATH = FILE_PATH.parent / "lwbs_significance_results.csv"

results_summary.to_csv(OUTPUT_PATH, index=False)

print("\nResults saved to:", OUTPUT_PATH)

,Analysis,Chi_Square,DF,P_Value,Significant_at_0.05
0,Triage Score vs LWBS (GEE),1.389978,4.0,0.845935,False
1,Wait Band vs LWBS (GEE),1.856430,3.0,0.602733,False
2,Wait Band vs LWBS (Pearson),1.856527,3.0,0.602712,False
3,Arrival Year vs LWBS (GEE),6.420616,6.0,0.377757,False
4,Arrival Hour vs LWBS (GEE),30.854285,23.0,0.126415,False
5,Wait Distribution: LWBS vs Non-LWBS,NaN,NaN,0.316247,False



Results saved to: C:\Users\praso\Downloads\healthcare-ed-throughput\docs\lwbs_significance_results.csv


In [12]:
print("Missing values in analysis variables:")
print(
    df[
        [
            "Visit_ID",
            "Patient_ID",
            "Arrival_Timestamp",
            "Disposition_Type",
            "Observed_Triage_Wait_Minutes",
            "Triage_Score"
        ]
    ].isna().sum()
)

print("\nDuplicate Visit IDs:", df["Visit_ID"].duplicated().sum())

print("\nWait-time range:")
print("Minimum:", df["Observed_Triage_Wait_Minutes"].min())
print("Maximum:", df["Observed_Triage_Wait_Minutes"].max())

print("\nDisposition counts:")
print(df["Disposition_Type"].value_counts())

Missing values in analysis variables:
Visit_ID                        0
Patient_ID                      0
Arrival_Timestamp               0
Disposition_Type                0
Observed_Triage_Wait_Minutes    0
Triage_Score                    0
dtype: int64

Duplicate Visit IDs: 0

Wait-time range:
Minimum: 0
Maximum: 59

Disposition counts:
Disposition_Type
Discharged    99845
Admitted      80283
LWBS          19872
Name: count, dtype: int64


In [14]:

import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, mannwhitneyu
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

# --------------------------------------------------
# 1. Load and prepare the corrected dataset
# --------------------------------------------------
PROJECT_ROOT = Path.cwd()
FILE_PATH = PROJECT_ROOT / "docs" / "ed_visits_timestamp_corrected.csv"

df = pd.read_csv(FILE_PATH)

df["Arrival_Timestamp"] = pd.to_datetime(
    df["Arrival_Timestamp"],
    format="%d-%m-%Y %H:%M",
    errors="coerce"
)

df["Observed_Triage_Wait_Minutes"] = pd.to_numeric(
    df["Observed_Triage_Wait_Minutes"], errors="coerce"
)

df["Triage_Score"] = pd.to_numeric(
    df["Triage_Score"], errors="coerce"
)

df["LWBS"] = (
    df["Disposition_Type"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("LWBS")
    .astype(int)
)

df["Arrival_Year"] = df["Arrival_Timestamp"].dt.year
df["Arrival_Hour"] = df["Arrival_Timestamp"].dt.hour

# Wait-time analysis: exclude zero placeholder values
df_valid_wait = df.loc[
    df["Observed_Triage_Wait_Minutes"].notna()
    & (df["Observed_Triage_Wait_Minutes"] > 0)
].copy()

df_valid_wait["Wait_Band"] = pd.cut(
    df_valid_wait["Observed_Triage_Wait_Minutes"],
    bins=[0, 15, 30, 45, 60],
    labels=["0-15", "16-30", "31-45", "46-60"],
    include_lowest=True
)

print("All visits:", len(df))
print("Valid wait-time records:", len(df_valid_wait))
print("Overall LWBS cases:", df["LWBS"].sum())
print(
    "LWBS cases with valid wait times:",
    df_valid_wait["LWBS"].sum()
)

# --------------------------------------------------
# 2. Helper function: Pearson chi-square test
# --------------------------------------------------
def chi_square_test(data, variable):
    table = pd.crosstab(data[variable], data["LWBS"])
    chi2, p, dof, _ = chi2_contingency(table)

    return {
        "Test": f"{variable} vs LWBS",
        "N": int(table.to_numpy().sum()),
        "Statistic": round(chi2, 4),
        "df": int(dof),
        "p_value": round(p, 6)
    }

results = []

# Categorical associations using all eligible visits
score_data = df.dropna(subset=["Triage_Score", "LWBS"]).copy()
score_data["Triage_Score"] = score_data["Triage_Score"].astype(str)
results.append(chi_square_test(score_data, "Triage_Score"))

year_data = df.dropna(subset=["Arrival_Year", "LWBS"]).copy()
year_data["Arrival_Year"] = year_data["Arrival_Year"].astype(str)
results.append(chi_square_test(year_data, "Arrival_Year"))

hour_data = df.dropna(subset=["Arrival_Hour", "LWBS"]).copy()
hour_data["Arrival_Hour"] = hour_data["Arrival_Hour"].astype(str)
results.append(chi_square_test(hour_data, "Arrival_Hour"))

# Wait-band test uses valid wait-time records only
wait_data = df_valid_wait.dropna(subset=["Wait_Band", "LWBS"]).copy()
results.append(chi_square_test(wait_data, "Wait_Band"))

# --------------------------------------------------
# 3. Mann-Whitney U test: valid wait times only
# --------------------------------------------------
lwbs_wait = df_valid_wait.loc[
    df_valid_wait["LWBS"] == 1,
    "Observed_Triage_Wait_Minutes"
]

non_lwbs_wait = df_valid_wait.loc[
    df_valid_wait["LWBS"] == 0,
    "Observed_Triage_Wait_Minutes"
]

u_stat, mw_p = mannwhitneyu(
    lwbs_wait,
    non_lwbs_wait,
    alternative="two-sided"
)

mann_whitney_result = {
    "Test": "Valid wait time: LWBS vs non-LWBS",
    "N": len(lwbs_wait) + len(non_lwbs_wait),
    "Statistic": round(float(u_stat), 4),
    "df": np.nan,
    "p_value": round(float(mw_p), 6),
}

results.append(mann_whitney_result)

# --------------------------------------------------
# 4. Display consolidated results
# --------------------------------------------------
results_df = pd.DataFrame(results)

results_df["Conclusion_at_0.05"] = np.where(
    results_df["p_value"] < 0.05,
    "Statistically significant",
    "Not statistically significant"
)

display(results_df)

# Save the validation results
OUTPUT_PATH = FILE_PATH.parent / "lwbs_statistical_validation.csv"
results_df.to_csv(OUTPUT_PATH, index=False)

print(f"\nResults saved to: {OUTPUT_PATH}")


All visits: 200000
Valid wait-time records: 188144
Overall LWBS cases: 19872
LWBS cases with valid wait times: 18692


,Test,N,Statistic,df,p_value,Conclusion_at_0.05
0,Triage_Score vs LWBS,200000,1.400200e+00,4.0,0.844160,Not statistically significant
1,Arrival_Year vs LWBS,200000,6.417700e+00,6.0,0.378063,Not statistically significant
2,Arrival_Hour vs LWBS,200000,3.061570e+01,23.0,0.132529,Not statistically significant
3,Wait_Band vs LWBS,188144,2.330600e+00,3.0,0.506682,Not statistically significant
4,Valid wait time: LWBS vs non-LWBS,188144,1.591636e+09,NaN,0.259916,Not statistically significant



Results saved to: C:\Users\praso\Downloads\healthcare-ed-throughput\docs\lwbs_statistical_validation.csv
